In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, classification_report, confusion_matrix, accuracy_score

import warnings
warnings.filterwarnings('ignore')

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
ANON_path = os.path.join(path,'Q1_data.csv')
df_ANON = pd.read_csv(ANON_path)

print(f"Dataset shape: {df_ANON.shape}")


In [ ]:
# Task 2: Write your code here:
df_ANON.head()

In [ ]:
# Task 3: Write your code here:
df_ANON.info()

In [ ]:
# Task 4: Write your code here:
df_ANON.describe()

In [ ]:
  # Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df_ANON['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Price Distribution')
plt.xlabel('Price')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df_ANON.drop(columns=['Order_ID'])

In [ ]:
# Task 2: Write your code here:
def check_missing_values(df_ANON):
  missing_values = df_ANON.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df_ANON)

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df_ANON):
  duplicates = df_ANON.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df_ANON.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_ANON)

In [ ]:
# Task 4: Write your code here:
categorical_cols = df_ANON.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))
#Label Encoding
from sklearn.preprocessing import LabelEncoder

label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df_ANON[col] = le.fit_transform(df_ANON[col])
  label_encoders[col] = le

df_ANON

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = df_ANON.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")

scaler = StandardScaler()
df_ANON[numerical_cols] = scaler.fit_transform(df_ANON[numerical_cols])
df_ANON.head()

In [ ]:
# Task 6: Write your code here:

In [ ]:
# Task 1: Write your code here:
X_reg = df_ANON.drop("Delivery_Time", axis=1).astype(float)
y_reg = df_ANON['Delivery_Time'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
feature_cols = df_ANON.drop(columns=['Delivery_Time'])
# Use previously generated random data (example: regression data)
X, y = X_reg.copy(), y_reg.copy()


# Define K-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Iterate through folds
for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
    # indexing for each fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    # print shapes
    print(f"Fold {fold}")
    print("  X_train shape:", X_train.shape)
    print("  X_test shape :", X_test.shape)
    print("  y_train shape:", y_train.shape)
    print("  y_test shape :", y_test.shape)
    print("-" * 30)

In [ ]:
from sklearn.preprocessing import StandardScaler

# Initialize the scaler
scaler_template = StandardScaler()
X_train_tpl, X_test_tpl, y_train_tpl, y_test_tpl = train_test_split(X, y, test_size=0.2, random_state=42,shuffle= True)
X_train_scaled_tpl = scaler_template.fit_transform(X_train_tpl)
X_test_scaled_tpl = scaler_template.transform(X_test_tpl)


In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Initialize the model
# Adjust hyperparameters like n_estimators, max_depth as needed
model_template = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)

# Train the model
model_template.fit(X_train_scaled_tpl, y_train_tpl)

print("Model trained!")


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error


# Make predictions on the test set
y_pred_tpl = model_template.predict(X_test_scaled_tpl)

# Calculate evaluation metrics
mae_tpl = mean_absolute_error(y_test_tpl, y_pred_tpl)

print(f"MAE:  ${mae_tpl:,.2f}")

In [ ]:
# Initialize K-Fold Cross-Validation
kfold_template = KFold(n_splits=5, shuffle=True, random_state=42) # n_splits determines number of folds

mae_scores_tpl = []

# Loop through each fold
for train_idx_tpl, val_idx_tpl in kfold_template.split(X_train_scaled_tpl):
    X_fold_train_tpl, X_fold_val_tpl = X_train_scaled_tpl[train_idx_tpl], X_train_scaled_tpl[val_idx_tpl]
    y_fold_train_tpl, y_fold_val_tpl = y_train_tpl.iloc[train_idx_tpl], y_train_tpl.iloc[val_idx_tpl]

    # Initialize and train model for the current fold (can be a new instance or the same)
    fold_model_template = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
    fold_model_template.fit(X_fold_train_tpl, y_fold_train_tpl)

    # Predict and evaluate for the current fold
    y_fold_pred_tpl = fold_model_template.predict(X_fold_val_tpl)

    mae_scores_tpl.append(mean_absolute_error(y_fold_val_tpl, y_fold_pred_tpl))

mae_scores_tpl = np.array(mae_scores_tpl)


print(f"{kfold_template.n_splits}-Fold CV Results:")
print(f"Average MAE:  ${mae_scores_tpl.mean():,.2f}")

In [ ]:
# Task 1: Write your code here:
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model_template.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'], color='purple')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()


In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: